In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
import os
import csv
import shutil
import soundfile as sf
import numpy as np

from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset


In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperConfig, WhisperProcessor
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# # ---- Config ----
MODEL_NAME = "Kennethdot/kasanoma_whisper"
ENCODER_LAYERS_TARGET = 4   # from 12
DECODER_LAYERS_TARGET = 4   # from 12
SAVE_PATH = "./whisper_small_depth4"

In [ ]:
parent = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
parent_config = parent.config

print(f"Parent encoder layers: {parent_config.encoder_layers}")
print(f"Parent decoder layers: {parent_config.decoder_layers}")
print(f"Parent total params: {sum(p.numel() for p in parent.parameters()):,}")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
def evenly_spaced_indices(n_total, n_keep):
    """Pick n_keep evenly spaced layer indices from n_total, always including first and last."""
    if n_keep == 1:
        return [n_total - 1]
    step = (n_total - 1) / (n_keep - 1)
    return [round(i * step) for i in range(n_keep)]


encoder_indices = evenly_spaced_indices(parent_config.encoder_layers, ENCODER_LAYERS_TARGET)
decoder_indices = evenly_spaced_indices(parent_config.decoder_layers, DECODER_LAYERS_TARGET)

print(f"Keeping encoder layers: {encoder_indices}")
print(f"Keeping decoder layers: {decoder_indices}")

# ---- Build the smaller config ----
student_config = WhisperConfig.from_pretrained(MODEL_NAME)
student_config.encoder_layers = ENCODER_LAYERS_TARGET
student_config.decoder_layers = DECODER_LAYERS_TARGET

# ---- Instantiate student with random weights, matching architecture shape ----
student = WhisperForConditionalGeneration(student_config)

# ---- Copy everything that isn't layer-indexed (embeddings, conv stem, norms, proj) ----
student.model.encoder.conv1.load_state_dict(parent.model.encoder.conv1.state_dict())
student.model.encoder.conv2.load_state_dict(parent.model.encoder.conv2.state_dict())
student.model.encoder.embed_positions.load_state_dict(parent.model.encoder.embed_positions.state_dict())
student.model.encoder.layer_norm.load_state_dict(parent.model.encoder.layer_norm.state_dict())

student.model.decoder.embed_tokens.load_state_dict(parent.model.decoder.embed_tokens.state_dict())
student.model.decoder.embed_positions.load_state_dict(parent.model.decoder.embed_positions.state_dict())
student.model.decoder.layer_norm.load_state_dict(parent.model.decoder.layer_norm.state_dict())

# proj_out / lm_head — usually tied to decoder embeddings, but copy explicitly to be safe
if hasattr(parent, "proj_out"):
    student.proj_out.load_state_dict(parent.proj_out.state_dict())

# ---- Copy the selected encoder/decoder layers ----
for student_idx, parent_idx in enumerate(encoder_indices):
    student.model.encoder.layers[student_idx].load_state_dict(
        parent.model.encoder.layers[parent_idx].state_dict()
    )

for student_idx, parent_idx in enumerate(decoder_indices):
    student.model.decoder.layers[student_idx].load_state_dict(
        parent.model.decoder.layers[parent_idx].state_dict()
    )

print(f"Student total params: {sum(p.numel() for p in student.parameters()):,}")
print(f"Compression ratio: {sum(p.numel() for p in parent.parameters()) / sum(p.numel() for p in student.parameters()):.2f}x")

# ---- Save the warm-started (not yet fine-tuned) student ----
student.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)

print(f"Warm-started student saved to {SAVE_PATH}")
print("Next: fine-tune this checkpoint on your Project Kasa training data before benchmarking.")

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(SAVE_PATH)

In [9]:
# Calculate model memory footprint

model_size_bytes = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
)

model_size_mb = model_size_bytes / (1024 ** 2)

print(f"Model size: {model_size_mb:.2f} MB")

Model size: 543.64 MB


In [8]:
from datasets import load_from_disk
my_audio_dataset = load_from_disk("./processed_dataset")
my_audio_dataset.set_format("torch")

Loading dataset from disk:   0%|          | 0/98 [00:00<?, ?it/s]

In [9]:
print(my_audio_dataset['train'])

Dataset({
    features: ['input_features', 'labels', 'input_length'],
    num_rows: 50965
})


In [29]:
#@title Training Hyper Parameters
OUTPUT_DIR = './whisper_tuning_akan_depth' #@param
LOG_DIR = os.path.join(OUTPUT_DIR, 'logs')

LEARNING_RATE = 1e-5 #@param
MAX_EPOCHS = 20 #@param
WARMUP_STEPS = 100 #@param
# set this as short as possible for your data
MAX_GEN_LEN = 32 #@param
# if save steps is 0, only last and best model will be written
SAVE_STEPS = 1500 #@param

# see
# https://huggingface.co/docs/transformers/v4.46.2/en/main_classes/trainer#transformers.TrainingArguments
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=OUTPUT_DIR + '/logs',
    per_device_train_batch_size=8,      # increase if VRAM allows
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    fp16=True,
    num_train_epochs=MAX_EPOCHS,
    #
    lr_scheduler_type='constant_with_warmup',
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    #
    eval_strategy="steps",
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    generation_max_length=MAX_GEN_LEN,
    eval_steps=1500,
    metric_for_best_model="wer",
    greater_is_better=False,
    #
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    logging_steps=10,
    # report_to=["tensorboard"],
    load_best_model_at_end=True,
    #
    push_to_hub=False,
    remove_unused_columns=False,
    save_total_limit=3,
    dataloader_num_workers=0,      # parallel data loading
    dataloader_pin_memory=False,
    #eval_on_start=True,
)

In [5]:
def count_trainable_parameters(model):
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    return params

In [6]:
#@title define which parameters to update
#@markdown For personalizartion, we typically only want to update the encoder and projection layer.
#@markdown Updating the decoder layer may lead to overfitting.
UPDATE_ENCODER = True #@param{type: 'boolean'}
UPDATE_DECODER = False #@param{type: 'boolean'}
UPDATE_PROJ = True #@param{type: 'boolean'}
model.model.encoder.requires_grad_(UPDATE_ENCODER)
model.model.decoder.requires_grad_(UPDATE_DECODER)
model.proj_out.requires_grad_(UPDATE_PROJ)


print('encoder params to update/total:', count_trainable_parameters(model.model.encoder), model.model.encoder.num_parameters())
print('decoder parans to update/total:', count_trainable_parameters(model.model.decoder), model.model.decoder.num_parameters())

print('overall # trainable parameters:', count_trainable_parameters(model))
print('overall # model parameters:', model.model.num_parameters())

encoder params to update/total: 31457280 31457280
decoder parans to update/total: 39832320 77978880
overall # trainable parameters: 71289600
overall # model parameters: 109436160


In [32]:
#@title Define Trainer
import evaluate
metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=my_audio_dataset["train"],
    eval_dataset=my_audio_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)


In [33]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [34]:
trainer.train()

  0%|          | 0/31840 [00:00<?, ?it/s]

{'loss': 0.9104, 'grad_norm': 8.626701354980469, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.01}
{'loss': 0.8472, 'grad_norm': 7.655413627624512, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.01}
{'loss': 0.8652, 'grad_norm': 7.196207523345947, 'learning_rate': 3e-06, 'epoch': 0.02}
{'loss': 0.8905, 'grad_norm': 7.750959873199463, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.03}
{'loss': 1.0313, 'grad_norm': inf, 'learning_rate': 4.9000000000000005e-06, 'epoch': 0.03}
{'loss': 1.0028, 'grad_norm': 7.844603538513184, 'learning_rate': 5.9e-06, 'epoch': 0.04}
{'loss': 0.9113, 'grad_norm': 7.941591262817383, 'learning_rate': 6.9e-06, 'epoch': 0.04}
{'loss': 0.9072, 'grad_norm': 7.3485846519470215, 'learning_rate': 7.9e-06, 'epoch': 0.05}
{'loss': 0.8728, 'grad_norm': 7.966971397399902, 'learning_rate': 8.900000000000001e-06, 'epoch': 0.06}
{'loss': 0.8271, 'grad_norm': 7.32655143737793, 'learning_rate': 9.9e-06, 'epoch': 0.06}
{'loss': 0.8027, 'grad_norm': 8.21697425842

  0%|          | 0/540 [00:00<?, ?it/s]

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'suppress_tokens': []}


{'eval_loss': 0.40731891989707947, 'eval_wer': 69.58383660920171, 'eval_runtime': 265.9997, 'eval_samples_per_second': 8.117, 'eval_steps_per_second': 2.03, 'epoch': 0.94}


...

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'suppress_tokens': []}


{'eval_loss': 0.09433049708604813, 'eval_wer': 60.40408476995718, 'eval_runtime': 261.0589, 'eval_samples_per_second': 8.27, 'eval_steps_per_second': 2.068, 'epoch': 15.07}


c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': 0.0014, 'grad_norm': 0.14436505734920502, 'learning_rate': 1e-05, 'epoch': 15.07}
{'loss': 0.0017, 'grad_norm': 0.13579092919826508, 'learning_rate': 1e-05, 'epoch': 15.08}
{'loss': 0.0008, 'grad_norm': 2.0021088123321533, 'learning_rate': 1e-05, 'epoch': 15.09}
{'loss': 0.0012, 'grad_norm': 0.550700306892395, 'learning_rate': 1e-05, 'epoch': 15.09}
{'loss': 0.0006, 'grad_norm': 0.038287121802568436, 'learning_rate': 1e-05, 'epoch': 15.1}
{'loss': 0.0011, 'grad_norm': 0.06862913817167282, 'learning_rate': 1e-05, 'epoch': 15.11}
{'loss': 0.0008, 'grad_norm': 0.06225175783038139, 'learning_rate': 1e-05, 'epoch': 15.11}
{'loss': 0.0015, 'grad_norm': 0.041889794170856476, 'learning_rate': 1e-05, 'epoch': 15.12}
{'loss': 0.0008, 'grad_norm': 0.44930362701416016, 'learning_rate': 1e-05, 'epoch': 15.12}
{'loss': 0.0015, 'grad_norm': 0.5530384182929993, 'learning_rate': 1e-05, 'epoch': 15.13}
{'loss': 0.0009, 'grad_norm': 0.4975525438785553, 'learning_rate': 1e-05, 'epoch': 15.14}
{'l

KeyboardInterrupt: 

In [35]:
print('evaluating best model after fine-tuning, language:', None)
# Let Whisper handle multilingual input 
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

results = trainer.evaluate(my_audio_dataset["validation"].shuffle(seed=42).select(range(10)))
print(results)

evaluating best model after fine-tuning, language: None


  0%|          | 0/3 [00:00<?, ?it/s]

{'eval_loss': 0.03833552077412605, 'eval_wer': 65.85365853658537, 'eval_runtime': 1.5807, 'eval_samples_per_second': 6.326, 'eval_steps_per_second': 1.898, 'epoch': 15.39}
{'eval_loss': 0.03833552077412605, 'eval_wer': 65.85365853658537, 'eval_runtime': 1.5807, 'eval_samples_per_second': 6.326, 'eval_steps_per_second': 1.898, 'epoch': 15.394129649976456}


In [ ]:
output_dir = 'finetuned_whisper_model_depth_6' #@param {type: 'string'}
!mkdir -p {output_dir}

print('Saving model in:', output_dir)

# save model and processor, so we can later load as pretrained
save_model_dir = os.path.join(output_dir, 'saved_model')
trainer.model.save_pretrained(save_model_dir, safe_serialization=False)

# save processor also
save_processor_dir = os.path.join(output_dir, 'saved_processor')
processor.save_pretrained(save_processor_dir, safe_serialization=False)

A subdirectory or file -p already exists.
Error occurred while processing: -p.
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'suppress_tokens': []}


Saving model in: finetuned_whisper_model_depth_6


[]

: 